<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B03%5D%20-%20Algoritmos_Alternativos_Clasificacion/%5B01%5D%20-%20Notebooks/E7_Modelos_alternativos_NaiveBayes_SVM_Wine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelos Alternativos - Clasificación con Naive Bayes y SVM

## Introducción

En este notebook vamos a trabajar una sesión centrada en **modelos alternativos de clasificación**, con foco en dos familias muy importantes:

- **Naive Bayes**
- **Support Vector Machines (SVM)**

El objetivo no es entrenar veinte algoritmos “por si acaso”, sino **entender bien dos enfoques distintos**, sus supuestos, sus fortalezas, sus limitaciones y cómo se comportan sobre un problema real de clasificación.

Para ello utilizaremos el dataset **Wine** de `scikit-learn`, un dataset educativo clásico que contiene variables químicas de vinos clasificados en **tres clases**. Es una muy buena elección para esta sesión porque:

- todas las variables predictoras son numéricas,
- permite trabajar clasificación multiclase,
- encaja muy bien con **Gaussian Naive Bayes**,
- y además permite ilustrar de forma clara el efecto del escalado y del kernel en **SVM**.

## Objetivos

**O1.** Comprender el problema de clasificación y la estructura del dataset.  
**O2.** Analizar visualmente los datos antes de modelar.  
**O3.** Entender el funcionamiento de **Gaussian Naive Bayes** y sus supuestos.  
**O4.** Entender el funcionamiento de **SVM**, el margen máximo y el papel de los kernels.  
**O5.** Comparar ambos enfoques con métricas, matrices de confusión y visualizaciones.  
**O6.** Ajustar hiperparámetros de SVM y extraer conclusiones razonadas.

## Dataset

Trabajaremos con el dataset **Wine** cargado desde `sklearn.datasets`.

La variable objetivo representa tres tipos de vino. Las variables de entrada son mediciones físico-químicas como:

- alcohol
- malic_acid
- flavanoids
- color_intensity
- proline
- entre otras

## Idea de la sesión

Vamos a seguir este recorrido:

1. Carga y exploración del dataset  
2. Análisis visual de las variables  
3. Preparación de datos  
4. Modelo **Gaussian Naive Bayes**  
5. Modelo **SVM lineal**  
6. Modelo **SVM con kernel RBF**  
7. Comparación de resultados  
8. Ajuste de hiperparámetros  
9. Interpretación final

In [ ]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)
from sklearn.decomposition import PCA
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

print("Librerías cargadas correctamente.")

## 1. Carga del dataset

Cargamos el dataset `Wine` desde `sklearn` y lo convertimos en un `DataFrame` para trabajar con mayor comodidad.

In [ ]:
# ============================================================
# CARGA DEL DATASET
# ============================================================

wine = load_wine()

df = pd.DataFrame(wine.data, columns=wine.feature_names)
df["target"] = wine.target
df["target_name"] = df["target"].map({
    0: wine.target_names[0],
    1: wine.target_names[1],
    2: wine.target_names[2]
})

print("Dimensiones del dataset:", df.shape)
display(df.head())

## 2. Inspección inicial

Antes de modelar, toca entender el terreno:

- tamaño del dataset,
- tipos de dato,
- nulos,
- equilibrio de clases,
- distribución general de variables.


In [ ]:
# ============================================================
# INFORMACIÓN GENERAL
# ============================================================

display(df.info())
display(df.describe().T)

print("Valores nulos por columna:")
display(df.isnull().sum())

print("Número de duplicados:", df.duplicated().sum())

In [ ]:
# ============================================================
# DISTRIBUCIÓN DE CLASES
# ============================================================

class_counts = df["target_name"].value_counts().sort_index()
display(class_counts)

plt.figure(figsize=(8, 5))
plt.bar(class_counts.index, class_counts.values)
plt.title("Distribución de clases")
plt.xlabel("Clase")
plt.ylabel("Frecuencia")
plt.show()

### Lectura rápida

- El problema es **multiclase**.
- Las clases están **bastante equilibradas**.
- No parece que una única clase vaya a dominar artificialmente las métricas.

## 3. Análisis exploratorio de datos (EDA)

Vamos a observar:

- la distribución de las variables,
- diferencias de escala,
- posibles outliers,
- relaciones entre variables,
- y separación visual entre clases.

Esto es especialmente importante aquí porque:

- **Naive Bayes** hace supuestos probabilísticos sobre las variables,
- **SVM** es sensible a la geometría del espacio de entrada,
- y el **escalado** puede cambiar bastante el comportamiento del modelo.

In [ ]:
# ============================================================
# VARIABLES PREDICTORAS Y OBJETIVO
# ============================================================

X = df.drop(columns=["target", "target_name"])
y = df["target"]

print("Número de variables predictoras:", X.shape[1])
print("Clases:", wine.target_names)

In [ ]:
# ============================================================
# HISTOGRAMAS
# ============================================================

X.hist(figsize=(16, 12), bins=20)
plt.suptitle("Histogramas de las variables numéricas", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BOXPLOTS
# ============================================================

plt.figure(figsize=(16, 7))
plt.boxplot([X[col] for col in X.columns], labels=X.columns, vert=True)
plt.xticks(rotation=90)
plt.title("Boxplots de las variables")
plt.ylabel("Valor")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# MATRIZ DE CORRELACIÓN
# ============================================================

corr = X.corr()

plt.figure(figsize=(12,10))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5
)

plt.title("Matriz de correlación")
plt.tight_layout()
plt.show()

### Comentario

Aquí ya podemos intuir algunas cosas:

- hay variables en **escalas muy distintas**,
- hay relaciones entre variables,
- y probablemente el dataset **no sea perfectamente lineal** en todas las direcciones.

Eso sugiere dos ideas relevantes:

1. **SVM** probablemente agradecerá el escalado.  
2. **Gaussian Naive Bayes** puede funcionar razonablemente bien, pero su supuesto de independencia entre variables no será perfecto.



In [ ]:
# ============================================================
# VARIABLES CLAVE POR CLASE
# ============================================================

variables_clave = ["alcohol", "flavanoids", "color_intensity", "proline"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(variables_clave):
    data_by_class = [
        df.loc[df["target"] == clase, col]
        for clase in sorted(df["target"].unique())
    ]
    axes[i].boxplot(
        data_by_class,
        labels=[wine.target_names[c] for c in sorted(df["target"].unique())]
    )
    axes[i].set_title(f"{col} por clase")
    axes[i].set_xlabel("Clase")
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

## 4. Visualización con PCA

Como tenemos muchas variables, una forma útil de visualizar la estructura global es proyectar los datos en dos componentes principales.

Esto **no sustituye al modelo real**, pero ayuda muchísimo a explicar:

- si las clases parecen separables,
- si esa separación parece lineal,
- y por qué un SVM con kernel puede tener sentido.

In [ ]:
# ============================================================
# PCA PARA VISUALIZAR
# ============================================================

scaler_temp = StandardScaler()
X_scaled_temp = scaler_temp.fit_transform(X)

pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_scaled_temp)

plt.figure(figsize=(9, 7))

for clase in sorted(y.unique()):
    mask = y == clase
    plt.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        s=60,
        label=wine.target_names[clase]
    )

plt.title("Visualización del dataset en 2 componentes principales")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()

print("Varianza explicada por PC1 y PC2:", pca_2d.explained_variance_ratio_)
print("Varianza acumulada:", pca_2d.explained_variance_ratio_.sum())

### Pregunta de reflexión

A partir de esta proyección, ¿dirías que el problema parece:

- fácilmente lineal,
- parcialmente lineal,
- o con fronteras más complejas?

Esta pregunta prepara muy bien el terreno para comparar:

- **Gaussian Naive Bayes**
- **SVM lineal**
- **SVM con kernel RBF**

## 5. Train/Test split y preparación

Vamos a separar entrenamiento y prueba. Además, prepararemos dos enfoques:

- **Gaussian Naive Bayes**
- **SVM**

Importante:

- **SVM necesita escalado sí o sí** en casi cualquier caso serio.
- **Gaussian Naive Bayes** no depende tanto de la escala para funcionar, pero comparar versiones escaladas y no escaladas también puede ser interesante.

In [ ]:
# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Shape train:", X_train.shape, y_train.shape)
print("Shape test :", X_test.shape, y_test.shape)

In [ ]:
# ============================================================
# ESCALADO
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Escalado aplicado correctamente.")

## 6. Naive Bayes: intuición y entrenamiento

En esta sesión utilizaremos **Gaussian Naive Bayes**, una versión de Naive Bayes adecuada para variables numéricas continuas.

### Idea central

Gaussian Naive Bayes asume que:

1. las variables son **condicionalmente independientes** dada la clase,  
2. y que cada variable sigue una distribución aproximadamente **normal** dentro de cada clase.

Esto es una simplificación bastante fuerte, pero muchas veces funciona sorprendentemente bien.

In [ ]:
# ============================================================
# MODELO GAUSSIAN NAIVE BAYES
# ============================================================

gnb = GaussianNB()
gnb.fit(X_train, y_train)

y_pred_gnb = gnb.predict(X_test)

print("Resultados GaussianNB")
print("Accuracy       :", round(accuracy_score(y_test, y_pred_gnb), 4))
print("Precision macro:", round(precision_score(y_test, y_pred_gnb, average="macro"), 4))
print("Recall macro   :", round(recall_score(y_test, y_pred_gnb, average="macro"), 4))
print("F1 macro       :", round(f1_score(y_test, y_pred_gnb, average="macro"), 4))

In [ ]:
# ============================================================
# MATRIZ DE CONFUSIÓN - GAUSSIAN NB
# ============================================================

cm_gnb = confusion_matrix(y_test, y_pred_gnb)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_gnb, display_labels=wine.target_names)
disp.plot(ax=ax)
plt.title("Matriz de confusión - Gaussian Naive Bayes")
plt.show()

print(classification_report(y_test, y_pred_gnb, target_names=wine.target_names))

## 7. ¿Qué está modelando Naive Bayes?

Una forma muy útil de explicar Gaussian Naive Bayes es visualizar distribuciones de variables por clase.

No estamos viendo “todo el modelo”, pero sí una parte importante de su lógica:
el algoritmo estima, para cada variable y cada clase, una distribución gaussiana.

In [ ]:
# ============================================================
# DISTRIBUCIONES POR CLASE PARA ENTENDER GNB
# ============================================================

variables_nb = ["alcohol", "flavanoids", "color_intensity", "proline"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(variables_nb):
    for clase in sorted(y.unique()):
        values = df.loc[df["target"] == clase, col]
        axes[i].hist(values, bins=15, alpha=0.5, label=wine.target_names[clase])
    axes[i].set_title(f"Distribución de {col} por clase")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frecuencia")
    axes[i].legend()

plt.tight_layout()
plt.show()

### Comentario

Si una variable muestra distribuciones bastante distintas entre clases, Naive Bayes puede extraer señal útil de ahí.

El problema es que el modelo **trata cada variable como si aportara información de forma independiente**, y en datasets reales eso rara vez se cumple del todo.

## 8. SVM lineal

Ahora pasamos a **SVM**.

### Idea central de SVM

SVM busca una frontera que separe las clases maximizando el **margen** entre ellas.  

Primero probaremos una versión **lineal**. Esto sirve para ver si una frontera recta en el espacio transformado por escalado puede capturar bien la estructura del problema.

In [ ]:
# ============================================================
# MODELO SVM LINEAL
# ============================================================

svm_linear = SVC(kernel="linear", C=1.0, probability=True, random_state=42)
svm_linear.fit(X_train_scaled, y_train)

y_pred_svm_linear = svm_linear.predict(X_test_scaled)

print("Resultados SVM lineal")
print("Accuracy       :", round(accuracy_score(y_test, y_pred_svm_linear), 4))
print("Precision macro:", round(precision_score(y_test, y_pred_svm_linear, average="macro"), 4))
print("Recall macro   :", round(recall_score(y_test, y_pred_svm_linear, average="macro"), 4))
print("F1 macro       :", round(f1_score(y_test, y_pred_svm_linear, average="macro"), 4))

In [ ]:
# ============================================================
# MATRIZ DE CONFUSIÓN - SVM LINEAL
# ============================================================

cm_svm_linear = confusion_matrix(y_test, y_pred_svm_linear)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_svm_linear, display_labels=wine.target_names)
disp.plot(ax=ax)
plt.title("Matriz de confusión - SVM lineal")
plt.show()

print(classification_report(y_test, y_pred_svm_linear, target_names=wine.target_names))

## 9. SVM con kernel RBF

Aquí entra la artillería interesante.

Cuando las clases no pueden separarse bien con una frontera lineal, SVM puede usar un **kernel** para trabajar como si los datos estuvieran en un espacio de mayor dimensión.

En esta sesión usamos el **kernel RBF (Radial Basis Function)**.

### Intuición del RBF

En términos simples:

- mide cercanía entre puntos,
- crea una representación más flexible,
- y permite que el modelo aprenda fronteras no lineales.

In [ ]:
# ============================================================
# MODELO SVM RBF
# ============================================================

svm_rbf = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42)
svm_rbf.fit(X_train_scaled, y_train)

y_pred_svm_rbf = svm_rbf.predict(X_test_scaled)

print("Resultados SVM RBF")
print("Accuracy       :", round(accuracy_score(y_test, y_pred_svm_rbf), 4))
print("Precision macro:", round(precision_score(y_test, y_pred_svm_rbf, average="macro"), 4))
print("Recall macro   :", round(recall_score(y_test, y_pred_svm_rbf, average="macro"), 4))
print("F1 macro       :", round(f1_score(y_test, y_pred_svm_rbf, average="macro"), 4))

In [ ]:
# ============================================================
# MATRIZ DE CONFUSIÓN - SVM RBF
# ============================================================

cm_svm_rbf = confusion_matrix(y_test, y_pred_svm_rbf)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_svm_rbf, display_labels=wine.target_names)
disp.plot(ax=ax)
plt.title("Matriz de confusión - SVM con kernel RBF")
plt.show()

print(classification_report(y_test, y_pred_svm_rbf, target_names=wine.target_names))

## 10. Comparación conjunta de Gaussian NB vs SVM

Vamos a poner los tres enfoques frente a frente:

- Gaussian Naive Bayes
- SVM lineal
- SVM con RBF

In [ ]:
# ============================================================
# TABLA COMPARATIVA DE MODELOS
# ============================================================

comparison_df = pd.DataFrame([
    {
        "Modelo": "Gaussian Naive Bayes",
        "Accuracy": accuracy_score(y_test, y_pred_gnb),
        "Precision_macro": precision_score(y_test, y_pred_gnb, average="macro"),
        "Recall_macro": recall_score(y_test, y_pred_gnb, average="macro"),
        "F1_macro": f1_score(y_test, y_pred_gnb, average="macro")
    },
    {
        "Modelo": "SVM lineal",
        "Accuracy": accuracy_score(y_test, y_pred_svm_linear),
        "Precision_macro": precision_score(y_test, y_pred_svm_linear, average="macro"),
        "Recall_macro": recall_score(y_test, y_pred_svm_linear, average="macro"),
        "F1_macro": f1_score(y_test, y_pred_svm_linear, average="macro")
    },
    {
        "Modelo": "SVM RBF",
        "Accuracy": accuracy_score(y_test, y_pred_svm_rbf),
        "Precision_macro": precision_score(y_test, y_pred_svm_rbf, average="macro"),
        "Recall_macro": recall_score(y_test, y_pred_svm_rbf, average="macro"),
        "F1_macro": f1_score(y_test, y_pred_svm_rbf, average="macro")
    }
]).sort_values(by="F1_macro", ascending=False)

display(comparison_df)

In [ ]:
# ============================================================
# VISUALIZACIÓN COMPARATIVA
# ============================================================

metrics = ["Accuracy", "Precision_macro", "Recall_macro", "F1_macro"]

for metric in metrics:
    plt.figure(figsize=(8, 5))
    ordered = comparison_df.sort_values(by=metric, ascending=False)
    plt.bar(ordered["Modelo"], ordered[metric])
    plt.title(f"Comparación de modelos - {metric}")
    plt.ylabel(metric)
    plt.ylim(0.7, 1.02)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

## 11. Validación cruzada

Una sola partición train-test puede darnos una foto algo sesgada.  
Por eso, vamos a repetir la comparación con **validación cruzada estratificada**.

In [ ]:
# ============================================================
# VALIDACIÓN CRUZADA
# ============================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_cv = {
    "GaussianNB": GaussianNB(),
    "SVM lineal": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="linear", C=1.0, random_state=42))
    ]),
    "SVM RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42))
    ])
}

cv_results = []

for name, model in models_cv.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="f1_macro")
    cv_results.append({
        "Modelo": name,
        "CV_mean_F1_macro": scores.mean(),
        "CV_std_F1_macro": scores.std(),
        "fold_scores": scores
    })

cv_df = pd.DataFrame(cv_results).sort_values(by="CV_mean_F1_macro", ascending=False)
display(cv_df[["Modelo", "CV_mean_F1_macro", "CV_std_F1_macro"]])

In [ ]:
# ============================================================
# BOXPLOT DE SCORES EN VALIDACIÓN CRUZADA
# ============================================================

scores_for_plot = [row["fold_scores"] for _, row in cv_df.iterrows()]
labels_for_plot = cv_df["Modelo"].tolist()

plt.figure(figsize=(8, 5))
plt.boxplot(scores_for_plot, labels=labels_for_plot)
plt.title("Distribución de F1 macro en validación cruzada")
plt.ylabel("F1 macro")
plt.tight_layout()
plt.show()

### Idea importante

Aquí no solo importa quién gana por media. También conviene observar:

- si un modelo es más **estable**,
- si otro depende más de la partición,
- o si las diferencias son pequeñas y no justifican elegir un modelo mucho más complejo.

## 12. Fronteras de decisión en 2D (PCA)

Para enseñar la diferencia entre modelos, una visualización muy útil consiste en reducir los datos a 2 componentes principales y dibujar las fronteras de decisión.

**Advertencia importante:**  
estas gráficas sirven para entender el comportamiento de los modelos, pero no sustituyen la evaluación real sobre el espacio completo de variables.

In [ ]:
# ============================================================
# DATOS REDUCIDOS A 2 COMPONENTES
# ============================================================

X_scaled_all = StandardScaler().fit_transform(X)
pca_vis = PCA(n_components=2)
X_2d = pca_vis.fit_transform(X_scaled_all)

print("Shape de la proyección 2D:", X_2d.shape)

In [ ]:
# ============================================================
# FUNCIÓN PARA DIBUJAR FRONTERAS DE DECISIÓN
# ============================================================

def plot_decision_boundary(model, X_plot, y_plot, title, class_names):
    model.fit(X_plot, y_plot)

    x_min, x_max = X_plot[:, 0].min() - 1, X_plot[:, 0].max() + 1
    y_min, y_max = X_plot[:, 1].min() - 1, X_plot[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.3)

    for clase in np.unique(y_plot):
        mask = y_plot == clase
        plt.scatter(
            X_plot[mask, 0],
            X_plot[mask, 1],
            s=50,
            label=class_names[clase]
        )

    plt.title(title)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend()
    plt.show()

In [ ]:
# ============================================================
# FRONTERAS DE DECISIÓN
# ============================================================

plot_decision_boundary(
    GaussianNB(),
    X_2d,
    y.to_numpy(),
    "Frontera de decisión - Gaussian Naive Bayes",
    wine.target_names
)

plot_decision_boundary(
    SVC(kernel="linear", C=1.0, random_state=42),
    X_2d,
    y.to_numpy(),
    "Frontera de decisión - SVM lineal",
    wine.target_names
)

plot_decision_boundary(
    SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42),
    X_2d,
    y.to_numpy(),
    "Frontera de decisión - SVM RBF",
    wine.target_names
)

### Qué deberías observar

- **Gaussian Naive Bayes** genera regiones basadas en distribuciones probabilísticas por clase.
- **SVM lineal** intenta separar con fronteras lineales.
- **SVM RBF** puede adaptarse a estructuras más complejas.

Estas visualizaciones ayudan muchísimo a conectar el modelo con la geometría del problema.

## 13. Sensibilidad de SVM a hiperparámetros

SVM no es magia; sus resultados dependen bastante de ciertos hiperparámetros.

### Parámetros clave

- **C**: controla el equilibrio entre margen amplio y errores de clasificación.
- **gamma**: en RBF, controla el alcance de la influencia de cada punto.

Vamos a ajustar ambos con `GridSearchCV`.

In [ ]:
# ============================================================
# GRID SEARCH PARA SVM RBF
# ============================================================

svm_rbf_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", probability=True, random_state=42))
])

param_grid_rbf = {
    "model__C": [0.1, 1, 5, 10, 20, 50],
    "model__gamma": ["scale", 0.01, 0.05, 0.1, 0.5, 1]
}

grid_rbf = GridSearchCV(
    estimator=svm_rbf_pipe,
    param_grid=param_grid_rbf,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1
)

grid_rbf.fit(X_train, y_train)

print("Mejores parámetros SVM RBF:")
print(grid_rbf.best_params_)
print("Mejor score CV:", round(grid_rbf.best_score_, 4))

In [ ]:
# ============================================================
# EVALUACIÓN DEL MEJOR SVM RBF
# ============================================================

best_svm_rbf = grid_rbf.best_estimator_
y_pred_best_rbf = best_svm_rbf.predict(X_test)

print("Resultados del mejor SVM RBF ajustado")
print("Accuracy       :", round(accuracy_score(y_test, y_pred_best_rbf), 4))
print("Precision macro:", round(precision_score(y_test, y_pred_best_rbf, average="macro"), 4))
print("Recall macro   :", round(recall_score(y_test, y_pred_best_rbf, average="macro"), 4))
print("F1 macro       :", round(f1_score(y_test, y_pred_best_rbf, average="macro"), 4))

In [ ]:
# ============================================================
# MATRIZ DE CONFUSIÓN - BEST SVM RBF
# ============================================================

cm_best_rbf = confusion_matrix(y_test, y_pred_best_rbf)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_best_rbf, display_labels=wine.target_names)
disp.plot(ax=ax)
plt.title("Matriz de confusión - Mejor SVM RBF ajustado")
plt.show()

## 14. ROC multiclase

Como el problema es multiclase, podemos representar curvas ROC usando una estrategia **one-vs-rest** para comparar:

- Gaussian Naive Bayes
- SVM lineal
- SVM RBF ajustado

Esto añade una capa de análisis más avanzada y muy útil en clase.

In [ ]:
# ============================================================
# ROC MULTICLASE
# ============================================================

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
n_classes = y_test_bin.shape[1]

models_for_roc = {
    "GaussianNB": gnb,
    "SVM lineal": svm_linear,
    "Best SVM RBF": best_svm_rbf
}

for model_name, model in models_for_roc.items():
    if model_name == "GaussianNB":
        y_score = model.predict_proba(X_test)
    elif model_name == "SVM lineal":
        y_score = model.predict_proba(X_test_scaled)
    else:
        y_score = model.predict_proba(X_test)

    plt.figure(figsize=(8, 6))

    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"Clase {wine.target_names[i]} (AUC = {roc_auc:.3f})")

    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.title(f"Curvas ROC multiclase - {model_name}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.show()

## 15. Conclusiones

Después de todo el análisis, podemos sacar conclusiones.

### Sobre Gaussian Naive Bayes

- es rápido,
- simple,
- fácil de entrenar,
- y puede rendir sorprendentemente bien incluso si sus supuestos no se cumplen del todo.

Pero tiene limitaciones claras:

- asume independencia condicional entre variables,
- simplifica bastante la realidad,
- y puede quedarse corto si la frontera real entre clases es compleja.

### Sobre SVM

- **SVM lineal** es una opción elegante cuando la separación es aproximadamente lineal.
- **SVM con kernel RBF** ofrece mayor flexibilidad para capturar relaciones no lineales.
- El rendimiento de SVM depende mucho del **escalado** y de los hiperparámetros.

### Idea final importante

Se trata de entender:

- qué asume cada modelo,
- cómo representa el problema,
- cuándo conviene uno u otro,
- y cómo justificar la elección.

Eso es bastante más valioso que memorizar tres nombres de algoritmos y mirar el accuracy con fe ciega.